# TP53 Gene Mutation Prediction## METABRIC Breast Cancer RNA Expression Data### AI3013 Machine Learning Course Project**All models implemented from scratch using only NumPy, Pandas, and Matplotlib.**

## 1. Environment Setup

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt


## 2. Load Data

In [ ]:
from dataset.data_loader import load_tp53_data

X, y = load_tp53_data("dataset/METABRIC_RNA_Mutation.csv")


## 3. Train/Test Split & Standardization80% training / 20% test. Z-score: (x - mu) / sigma.Test set uses training mean and std to prevent data leakage.

In [ ]:
from dataset.preprocessing import custom_train_test_split, custom_standard_scaler

X_train, X_test, y_train, y_test = custom_train_test_split(X, y, test_size=0.2)
X_train, X_test = custom_standard_scaler(X_train, X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train pos/neg: {np.sum(y_train==1)}/{np.sum(y_train==0)}')


## 4. Run All Experiments5 comparison experiments + regularization paths + learning curves:

In [ ]:
from experiment import run_all_experiments

data = run_all_experiments(X_train, X_test, y_train, y_test)


## 5. Hold-out Test Results

In [ ]:
results = data["results"]
X_n_features = X_train.shape[1]

print(f"{'Model':30s}  {'Acc':>7s}  {'F1':>7s}  {'Prec':>7s}  {'Rec':>7s}  {'Features':>12s}")
print('-' * 85)
for name, (acc, f1, prec, rec, _, feat) in results.items():
    feat_str = f'{feat}d' if isinstance(feat, int) else str(feat)
    print(f'{name:30s}  {acc:7.4f}  {f1:7.4f}  {prec:7.4f}  {rec:7.4f}  {feat_str:>12s}')

majority = max(np.mean(y_test == 0), np.mean(y_test == 1))
print(f"{'Majority baseline':30s}  {majority:7.4f}  {'-':>7s}  {'-':>7s}  {'-':>7s}  {'-':>12s}")


## 6. 5-Fold Cross-Validation (mean +/- std)

In [ ]:
cv_results = data["cv_results"]

print(f"{'Model':30s}  {'Acc':>15s}  {'F1':>15s}  {'Prec':>15s}  {'Rec':>15s}")
print('-' * 85)
for name, cv in cv_results.items():
    print(f"{name:30s}  {cv['accuracy'][0]:.4f}±{cv['accuracy'][1]:.4f}  "
          f"{cv['f1'][0]:.4f}±{cv['f1'][1]:.4f}  "
          f"{cv['precision'][0]:.4f}±{cv['precision'][1]:.4f}  "
          f"{cv['recall'][0]:.4f}±{cv['recall'][1]:.4f}")


## 7. Key Findings

In [ ]:
svm_rbf_acc = results["SVM RBF"][0]
lr_l1_acc = results["LR + L1"][0]
svm_lin_acc = results["SVM Linear"][0]
l1_indices = data["l1_indices"]

print(f'SVM RBF  ({svm_rbf_acc:.4f}) > SVM Linear ({svm_lin_acc:.4f})  ' +
      f'-> Delta = +{(svm_rbf_acc - svm_lin_acc)*100:.1f}%  (nonlinear structure detected)')
print(f'SVM RBF  ({svm_rbf_acc:.4f}) > LR + L1    ({lr_l1_acc:.4f})  ' +
      f'-> Delta = +{(svm_rbf_acc - lr_l1_acc)*100:.1f}%  (nonlinear > sparse linear)')
print(f'LR + L1 uses only {len(l1_indices)}/{X_n_features} features but achieves {lr_l1_acc:.4f}  ' +
      '(interpretability vs. accuracy trade-off)')


## 8. Visualizations

In [ ]:
import visualization as viz


### 8.1 Model Performance Comparison

In [ ]:
viz.plot_metrics_comparison(results, cv_results)
plt.show()


### 8.2 Cost Convergence Curves

In [ ]:
viz.plot_cost_curves(data["cost_data"])
plt.show()


### 8.3 PCA Cumulative Explained VarianceHorizontal dashed line = 95% threshold (312 components selected from 489).

In [ ]:
viz.plot_pca_variance(data["pca"].explained_variance_ratio_)
plt.show()


### 8.4 L1 Regularization - Top 30 Selected GenesBlue = positive association with TP53 mutation; Coral = negative association.Only 106/489 genes retained by L1.

In [ ]:
viz.plot_l1_feature_weights(data["l1_indices"], data["l1_weights"], X_n_features)
plt.show()


### 8.5 Regularization Paths - CV Accuracy vs. Lambda

In [ ]:
for penalty, (lambda_values, cv_scores) in data["reg_path_data"].items():
    viz.plot_regularization_path(lambda_values, cv_scores, penalty.upper())
    plt.show()


### 8.6 Learning Curves - Overfitting/Underfitting AnalysisGap between training accuracy (blue) and CV validation accuracy (red) indicates overfitting.

In [ ]:
for name, (train_sizes, train_scores, val_scores) in data["lc_data"].items():
    viz.plot_learning_curve(train_sizes, train_scores, val_scores, name)
    plt.show()


## 9. Summary| Model | Key Feature | Best For ||---|---|---|| LR + L2 (Ridge) | All features, small weights | Baseline linear model || LR + L1 (Lasso) | **106/489 features** selected | Interpretability, gene discovery || PCA + LR + L2 | 312 PCs (95% variance) | Dimensionality reduction study || SVM Linear | Max-margin, class-balanced | Robust linear classifier || SVM RBF | **Best accuracy**, nonlinear kernel | Captures nonlinear decision boundaries |---*AI3013 Machine Learning Group Project*